In [1]:
!pip install langchain langchain-community chromadb tiktoken pypdf sentence-transformers langchain-openai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3

In [2]:
from google.colab import files
import os

os.makedirs("docs", exist_ok=True)

uploaded = files.upload()  # File picker khulega

for filename in uploaded.keys():
    os.rename(filename, f"docs/{filename}")
    print(f"✅ File save ho gayi: {filename}")

Saving Pakistan_Air_Force.pdf to Pakistan_Air_Force.pdf
✅ File save ho gayi: Pakistan_Air_Force.pdf


In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

pdf_file = [f for f in os.listdir("docs") if f.endswith(".pdf")][0]
loader = PyPDFLoader(f"docs/{pdf_file}")
documents = loader.load()

# ✅ Chunks bade kar diye
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # 500 se 1000 kar diya
    chunk_overlap=150  # 50 se 150 kar diya
)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks ban gaye")

✅ 228 chunks ban gaye


In [42]:
import shutil
import os

# Sab kuch delete karo
for folder in ["./chroma_db", "./chroma_db_new", "./chroma_db2"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"✅ {folder} delete ho gaya!")

print("✅ Clean slate ready!")

✅ Clean slate ready!


In [44]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Bilkul naye folder mein banao
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db2"  # ← naya naam
)

print("✅ Naya vector store ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Naya vector store ready!


In [45]:
!pip install langchain-groq -q
print("✅ Done!")

✅ Done!


In [55]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# ⚠️ Yahan apni GROQ key paste karo - gsk_ se shuru hogi
# Apni notebook mein key ki jagah yeh likho
GROQ_KEY = "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"  # ← key hata do  # ← apni real key yahan likho

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Prompt
prompt_template = PromptTemplate(
    template="""You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say "Mujhe is baare mein information nahi hai."

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

# Groq LLM
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0,
    api_key=GROQ_KEY
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Naya chain
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ Groq Chatbot ready hai!")

✅ Groq Chatbot ready hai!


In [56]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ✅ Smarter retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)

# ✅ Behtar prompt
prompt_template = PromptTemplate(
    template="""You are an expert assistant specifically about Pakistan Air Force.
Your job is to give detailed, accurate, and well-structured answers.

Instructions:
- Answer in detail using ALL relevant information from context
- If dates or numbers are mentioned, include them
- Structure your answer in points if multiple facts exist
- If context has partial info, share what you know
- Only say "information nahi hai" if context has absolutely nothing

Context:
{context}

Question: {question}

Detailed Answer:""",
    input_variables=["context", "question"]
)

# ✅ LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=1024,
    api_key=GROQ_KEY
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ Improved chatbot ready!")

✅ Improved chatbot ready!


In [57]:
from ipywidgets import widgets
from IPython.display import display, clear_output

text_input = widgets.Text(
    placeholder="Pakistan Air Force ke baare mein poochho...",
    layout=widgets.Layout(width="70%")
)
button = widgets.Button(
    description="Poochho",
    button_style="primary"
)
output = widgets.Output()

def on_ask(b):
    with output:
        clear_output()
        question = text_input.value.strip()
        if question:
            print(f"🧑 Aap: {question}\n")
            print("⏳ Soch raha hun...\n")
            docs = retriever.invoke(question)
            result = qa_chain.invoke(question)
            print(f"🤖 Bot:\n{result}")
            print("\n" + "─"*50)
            print("📄 Sources:")
            for i, doc in enumerate(docs[:3], 1):
                print(f"\n[{i}] Page {doc.metadata.get('page', '?') + 1}:")
                print(f"{doc.page_content[:150]}...")

button.on_click(on_ask)
display(widgets.HBox([text_input, button]), output)

Output()